In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_reviews = spark.table(f"{catalogo}.{esquema_source}.spotify_reviews")

In [0]:
df_reviews = df_reviews.dropna(subset=["Review"])

In [0]:
df_reviews_transformed = df_reviews.withColumn(
    "Time_submitted", F.to_timestamp(col("Time_submitted"), "yyyy-MM-dd HH:mm:ss")
).withColumn(
    "sentiment_category",
    when(col("Rating") >= 4, lit("Positivo"))\
    .when(col("Rating") == 3, lit("Neutral"))\
    .otherwise(lit("Negativo"))
).withColumn(
    "processed_date", current_timestamp()
)

In [0]:
df_final_reviews = df_reviews_transformed.select(
    "Time_submitted", "Review", "Rating", "Total_thumbsup", "Reply", 
    "sentiment_category", "ingestion_date", "processed_date"
)

In [0]:
df_final_reviews.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.spotify_reviews_clean")